# 06-1 · 깔끔한 결과 추출 — virtual image · fluctuation · 상별 위치+평균 NBD

nb6(탐색용)는 그대로 두고, **최종 결과만 깔끔하게 뽑는 전용 노트북**. 순서:
1. **Virtual image** (BF/ADF/structural)
2. **Fluctuation map** (FC-STEM 켑스트럼 + NBD-FEM variance)
3. **예상 물질** — 링 패턴에서 / 회절 스팟에서
4. **[링 기반]** 각 상: 링으로 정한 실공간 위치 + 그 위치들의 **평균 NBD** (+다른 링도 맞나 교차확인)
5. **[피크 기반]** 각 상: 대표 스팟으로 정한 위치 + 그 위치들의 **평균 NBD**

모든 플롯 텍스트는 ASCII. 그림 PNG + CSV 저장. 라벨의 '예상 물질'은 후보이며 확정 아님(LiF만 교차확인).

## 0) 준비 — 로드·설정 (nb6와 동일 설정)

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)

DET_BIN    = 2            # 검출기 비닝(메모리/속도). 합성이면 1
# --- 취득 조건(기록용; 분해능/해석 근거) ---
#   200 kV(λ≈2.51pm), 반수렴각 α≈1.1 mrad(직접빔 원반 2.2 mrad), 프로브≈1.4 nm,
#   스캔 100x100, step 2 nm(≈200x200 nm²), 검출기 512x512, 전류 12 nA는 조리개前 스크린값(시료엔 pA급).
#   ★링 위치 불확도 ≈ α/λ ≈ 0.044 1/A, 링 넓어짐 ≈ 2α/λ ≈ 0.088 1/A → q분해능 바닥 ~0.09 1/A.
#   → d≈2A에서 Δd≈0.35A라 LiF/Li2O/Li3N(~2.0A) 분리 불가(물리적 한계). 매칭 tol은 이에 맞춰 ~0.045.
Q_UNIT_HINT= "1/nm"     # ★ dm4 단위가 1/nm (0.043888 1/nm/px). 1/A로 두면 거리 10배 틀림
Q_PER_PX   = 0.02         # ★합성데이터 전용 fallback(1/A). 실데이터(dm4)는 이 값 무시하고
                          #   메타데이터(0.043888 1/nm →/10×binning= 0.008778 1/A)를 자동 사용.
                          #   여기에 0.043888을 넣지 마세요(1/nm≠1/A, 10배 틀림).
N_JOBS     = 4            # 병렬 코어수. ★메모리 부족(커널 꺼짐)이면 2로(-1/-2=전코어=메모리多)
EWPC_OFFSET= 1.0          # log(I+offset) — log(0) 방지
WINDOW     = True         # 켑스트럼 전 Hann 창(가장자리 십자 아티팩트 억제)
K          = 3            # 켑스트럼 프로파일 NMF/PCA 성분 수 (elbow 스캔으로 확인 후 조정)
KMAX_SCAN  = 8            # 성분 수 스캔 범위(1..KMAX): elbow/누적분산으로 '진짜 몇 개'인지
CENTER     = None         # (cx,cy) 수동. None이면 무게중심
CENTER_ZOOM= 25           # 중심 확대 그림 반경(px). 중심 맞는지 확인용(작을수록 확대)
STRONG_FRAC= 0.20         # §2d: 산란 강한 상위 비율만 골라 정렬평균(약한 링 추출 시도)
GAMMA      = 0.35         # §2d 2D 표시 감마(<1이면 약한 링 강조; 사용자가 감마로 링 본 것 재현)
RING_Q_MANUAL = None      # 감마 이미지에서 링이 보이면 그 q(1/A)를 직접 입력 → calibration/분석에 사용
INCLUDE_SUBSTRATE = False # 기판에 Cu가 있으면 True(Cu/Cu2O/CuO 후보 포함). 이 시료는 Cu 없음 → False
SIG_Q      = 0.045        # 링 패턴 언믹싱용 q-공간 브로드닝(1/A)
SPOT_PROM  = 0.02         # §2j Bragg 스팟 검출 민감도(작을수록 많이). min_prominence_frac
SPOT_MINDIST = 3          # §2j 스팟 최소 간격(px)
N_SPOTS    = 6            # §2m 단일-스팟 암시야로 볼 밝은 스팟 개수
SPOT_AP    = 3            # §2m 스팟 조리개 반경(px)
SPOT_ZOOM  = 15           # §2m 스팟 확인용 확대 반경(px, 작을수록 확대)
# 알려진 Li 화합물 원자간 거리(Å, 결정 근사) — 켑스트럼 프로파일에 참고선(비정질은 다소 이동)
CEPSTRAL_REF = {"Li-F 2.01": 2.01, "Li-O 2.00": 2.00, "Li-N 1.94": 1.94,
                "Li-S 2.47": 2.47, "2nd~2.9": 2.9}   # 후보 5개의 최근접이웃(pair)만
# 영역별 RDF용 설정 (조성은 peak 위치엔 영향 적음 — 아는 원소로 대략)
CFG_RDF = fds.RDFConfig(composition={"Li":1,"O":1}, q_int_min=0.15, q_int_max=1.0,
                        r_min=1.0, r_max=8.0, dr=0.02, damping="lorch")
# ★ 캘리브레이션 고정(known-standard): 비정질 첫 링(FSDP)의 최근접이웃 거리를 아는 값으로 맞춤.
#   None이면 메타데이터 q_per_px 그대로 사용(가정 없음).
#   Li-음이온이 주성분이면 CALIB_R_TARGET=2.0 (Å)로 두면 RDF 첫 peak이 그 거리로 옴.
#   → 메타데이터 대비 큰 보정이 필요하면 카메라 길이/단위를 다시 확인하세요.
CALIB_R_TARGET = None     # None=메타데이터(dm 0.043888 1/nm) 그대로 신뢰(권장). 아는 거리로 강제할 때만 값(예 2.0)
EHRENFEST      = 1.23     # cepstral 간격→pair 근사(단원자/금속글래스 기준; 이온성 Li는 1.1~1.4로 변함). 비율엔 무관, 절대거리는 RDF가 정확
# --- 빈(진공) 위치 제외 ---
# ★ nb5와 마스크를 '똑같이' 하려면 EMPTY_ROWS/EMPTY_ROI를 nb5와 같은 값으로 두세요.
#   (지정하면 그 빈 영역의 산란 평균+3σ로 임계 = nb5와 동일 방법. None이면 자동 Otsu라 조금 다름)
MATERIAL_MASK = True      # 물질 없는(진공) 위치를 분석에서 제외
ERODE_EDGE    = 2         # 물질 마스크를 이만큼(px) 침식 → 얇은 '가장자리 상'(두께 효과) 제외, 벌크만 분석
HOT_THRESHOLD = 8.0       # 고정 hot/dead 픽셀 검출 민감도(작을수록 민감)
DENOISE_CUBE  = False     # True면 전체 큐브의 고정 bad 픽셀 수리(메모리 큼). False면 평균 패턴만 정리
EMPTY_ROWS    = 10        # 아래 N행이 빈 영역(임계 기준). nb5의 EMPTY_ROWS와 같게 맞추세요
EMPTY_ROI     = None      # (y0,y1,x0,x1)로 빈 영역 직접 지정(있으면 EMPTY_ROWS 무시)
# 거리 밴드(Å) — 각각 하나의 FC-STEM 이미지. 데이터에 맞게 조절(먼저 §3 프로파일 보고).
BANDS      = [(1.0,1.5),(1.5,2.0),(2.0,2.5),(2.5,3.0),(3.0,3.5),(3.5,4.0),(4.0,4.5),(4.5,5.0),(5.0,5.5),(5.5,6.0)]

def make_fcstem_cube(scan=(36,48), dp=(96,96), empty_rows=8, seed=0):
    '''합성: 좌=결정(스팟), 중=비정질(halo), 우=다른 비정질(halo2), 아래=빈영역.
    FC-STEM이 결정/비정질/빈영역을 구분하는지 확인용.'''
    rng=np.random.default_rng(seed); Sy,Sx=scan; H,W=dp
    yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/(2*2.0**2))
    def halo(r0,s=4.0): return np.exp(-(rr-r0)**2/(2*s**2))
    def spots(r0,n=6,s=1.6,amp=4.0):
        img=np.zeros((H,W))
        for k in range(n):
            a=2*np.pi*k/n; sx,sy=cx+r0*np.cos(a),cy+r0*np.sin(a)
            img+=amp*np.exp(-((xx-sx)**2+(yy-sy)**2)/(2*s**2))
        return img
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            if iy>=Sy-empty_rows: base=0.9*beam                      # 빈 영역(진공)
            elif ix<Sx//3:        base=beam+spots(22)+0.5*halo(22)   # 결정
            elif ix<2*Sx//3:      base=beam+1.2*halo(20)             # 비정질 A
            else:                 base=beam+1.2*halo(28)             # 비정질 B
            cube[iy,ix]=base+0.15*rng.standard_normal((H,W))
    return np.clip(cube,0,None)

if USE_SYNTHETIC:
    cube=fds.from_array(make_fcstem_cube(empty_rows=(EMPTY_ROWS or 8)), q_per_px=Q_PER_PX, name="synthetic-FCSTEM")
else:
    cube=fds.load(DM4_PATH, Q_UNIT_HINT)
    print("raw loaded shape:", cube.data.shape, "(ndim", cube.ndim, ")")
    if cube.ndim<3:
        raise ValueError(f"{cube.ndim}D — not a scan; 로더가 데이터셋 검색 후에도 2D면 직접 로드하세요.")
    if DET_BIN>1: cube=fds.bin_cube_detector(cube, DET_BIN)
scan=cube.scan_shape; dp=cube.dp_shape
QPP = cube.calibration.q_per_px or Q_PER_PX
DR  = fds.quefrency_per_px(dp[0], QPP)      # 켑스트럼 픽셀당 Å
print("cube:", cube.shape, "| scan:", scan, "| dp:", dp)
print(f"q_per_px = {QPP:.5g} 1/A/px  ->  cepstral dr = {DR:.4g} A/px,  r_max ~ {DR*(dp[0]//2):.1f} A")

# 표시용 맵 재배열(3D 스택 대비) + 저장 헬퍼
import math
def _mapshape(n):
    r=int(math.sqrt(n))
    while r>1 and n%r: r-=1
    return (r,n//r) if r>1 else (1,n)
MAP=tuple(scan) if len(scan)==2 else _mapshape(int(np.prod(scan)))
def as_map(v):
    v=np.asarray(v,float).ravel(); m=np.full(int(np.prod(MAP)),np.nan); m[:min(v.size,m.size)]=v[:m.size]; return m.reshape(MAP)
SAVE_DIR=(os.path.dirname(DM4_PATH)+"/nb6_1_outputs" if not USE_SYNTHETIC else "nb6_1_outputs")
os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,name): p=os.path.join(SAVE_DIR,name+".png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(name,header,rows):
    import csv; p=os.path.join(SAVE_DIR,name+".csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(header); w.writerows(rows)
    print("saved:",p)
print("outputs ->", os.path.abspath(SAVE_DIR))


## 0b) 중심 · 물질 마스크 (진공 제외)

In [ ]:

med=fds.median_pattern(cube); mx=cube.max_dp()
if CENTER is not None: cx,cy=CENTER
else: cx,cy=fds.center_of_mass(med, threshold=0.3)

# 물질 마스크 (빈 영역 제외)
if MATERIAL_MASK and len(scan)==2:
    empty=None
    if EMPTY_ROI is not None:
        empty=np.zeros(scan,bool); y0,y1,x0,x1=EMPTY_ROI; empty[y0:y1,x0:x1]=True
    elif EMPTY_ROWS:
        empty=np.zeros(scan,bool); empty[max(0,scan[0]-EMPTY_ROWS):,:]=True
    material=fds.material_mask(cube, center=(cx,cy), empty_mask=empty)
    if ERODE_EDGE > 0:                               # 얇은 가장자리(두께 효과) 제외 → 벌크만
        try:
            from scipy.ndimage import binary_erosion
            material = binary_erosion(material, iterations=int(ERODE_EDGE))
        except Exception: pass
else:
    material=np.ones(scan,bool)
KEEP=material.ravel()
def scatter(vec, fill=np.nan):   # 물질 전용 결과를 스캔 전체로 되돌림(빈 곳=fill)
    out=np.full(KEEP.size,fill,float); out[KEEP]=np.asarray(vec,float).ravel(); return out.reshape(scan)
print(f"center: ({cx:.1f},{cy:.1f}) | material positions: {int(KEEP.sum())}/{KEEP.size} ({100*KEEP.mean():.0f}%)")

fig,ax=plt.subplots(1,3,figsize=(13,4.2))
im=ax[0].imshow(np.log1p(med),cmap="magma"); ax[0].plot(cx,cy,"c+",ms=10); ax[0].set_title("median (log)"); ax[0].axis("off")
im=ax[1].imshow(np.log1p(mx),cmap="magma"); ax[1].plot(cx,cy,"c+",ms=10); ax[1].set_title("MAX projection (strongest signal)"); ax[1].axis("off")
ax[2].imshow(material,cmap="gray"); ax[2].set_title("material mask (white = analyzed)"); ax[2].axis("off")
plt.tight_layout(); save(fig,"00_median_max_material"); plt.show()
np.save(os.path.join(SAVE_DIR,"00_max_projection.npy"), np.asarray(mx))


## 0c) 캘리브레이션·중심빔 제거 → q_beam, 후보 상 설정

In [ ]:
mp = fds.average_pattern(cube, material)
qd, Id = fds.azimuthal_integrate(mp, (cx,cy), q_per_px=QPP)
_,__,fsdp_base,fsdp_res = fds.find_fsdp(qd, Id, q_lo=0.10, q_hi=0.95, return_curves=True)
bp=int(np.argmax(np.where(qd<0.15, fsdp_res, -np.inf)))
hi=int(np.searchsorted(qd,0.45))
q_beam=float(qd[bp+int(np.argmin(fsdp_res[bp:hi]))]) if hi>bp+1 else 0.15
q_ring, ring_conf = fds.find_fsdp(qd, Id, q_lo=max(q_beam,0.12), q_hi=0.95)
CANDIDATES=['LiF','Li2O','Li3N','Li2CO3','Li2S']
colors={'LiF':'#e41a1c','Li2O':'#377eb8','Li3N':'#4daf4a','Li2CO3':'#984ea3','Li2S':'#ff7f00'}
RING_TOL=0.045                      # 물리 q분해능(alpha/lambda~0.044 1/A)에 맞춘 매칭 허용오차
SPOT_NMAD=8                         # 스팟 검출 임계 = median+SPOT_NMAD*MAD (크게 하면 노이즈 더 제거, 진짜 Bragg만)
STRONG_RING={c:max(fds.COMPOUND_RINGS[c],key=lambda t:t[1])[0] for c in CANDIDATES}  # 상별 최강 링 d(A)
print(f'QPP={QPP:.6f} 1/A/px | q_beam={q_beam:.3f} (d={1/q_beam:.2f}A) | FSDP q_ring={q_ring:.3f} conf {ring_conf:.1f}')
print('상별 대표(최강) 링 d(A):',{c:round(d,2) for c,d in STRONG_RING.items()})
print(f'매칭 허용오차 RING_TOL={RING_TOL} 1/A (수렴각 분해능)')

## 1) Virtual image — BF / ADF / structural(두께정규화)

In [ ]:
bf =np.asarray(fds.bright_field(cube,center=(cx,cy)),float)
adf=np.asarray(fds.annular_dark_field(cube,center=(cx,cy)),float)
sm =np.asarray(fds.structural_map(cube,center=(cx,cy)),float)     # 링/총 = 두께 정규화(줄무늬 제거)
imgs=[(bf,'bright field'),(adf,'annular dark field'),(sm,'structural (ring/total, thickness-norm)')]
fig,ax=plt.subplots(1,3,figsize=(13,4.2))
for a,(img,t) in zip(ax,imgs):
    im=a.imshow(np.where(material,img,np.nan),cmap='inferno'); a.set_title(t,fontsize=9); a.axis('off'); plt.colorbar(im,ax=a,fraction=0.046)
plt.tight_layout(); save(fig,'01_virtual_images'); plt.show()
save_csv('01_virtual_images',['scan_index','BF','ADF','structural'],
         [[i,f'{bf.ravel()[i]:.4f}',f'{adf.ravel()[i]:.4f}',f'{sm.ravel()[i]:.5f}'] for i in range(bf.size)])

## 2) Fluctuation map — FC-STEM(켑스트럼) + NBD-FEM(회절 variance)

FC-STEM: 거리 밴드별 켑스트럼 변동 F(Rp) (논문 Fig 4). NBD-FEM: 회절 강도 variance/mean² (논문 Fig 5, 결정성).
밴드는 대표 몇 개만 — 필요시 `BANDS`/아래 q밴드 조절.

In [ ]:
# FC-STEM: 대표 거리 밴드 3개 (짧음/중간/긺)
FBANDS=[(1.0,2.0),(2.0,3.5),(3.5,5.5)]
fmaps=fds.fluctuation_multiband(cube, FBANDS, QPP, offset=EWPC_OFFSET, window=WINDOW, n_jobs=N_JOBS)
# NBD-FEM: 대표 q 밴드 3개에서 회절 variance/mean^2 (결정성=스팟)
QBANDS=[(0.25,0.40),(0.40,0.55),(0.55,0.75)]   # 1/A
yy,xx=np.mgrid[0:dp[0],0:dp[1]]; rr=np.hypot(xx-cx,yy-cy)*QPP
flat=cube._flat_patterns().reshape(-1,dp[0]*dp[1])
def var_map(qlo,qhi):
    idx=np.where(((rr>=qlo)&(rr<qhi)).ravel())[0]
    sub=flat[:,idx].astype(np.float32)
    return (sub.var(1)/(sub.mean(1)**2+1e-6)).reshape(scan)
vmaps=[var_map(a,b) for a,b in QBANDS]

fig,ax=plt.subplots(2,3,figsize=(14,8))
for j,(a,b) in enumerate(FBANDS):
    m=as_map(np.asarray(fmaps[j],float))
    im=ax[0][j].imshow(np.where(material,m,np.nan),cmap='viridis'); ax[0][j].set_title(f'FC-STEM {a}-{b} A',fontsize=9); ax[0][j].axis('off'); plt.colorbar(im,ax=ax[0][j],fraction=0.046)
for j,(a,b) in enumerate(QBANDS):
    vm=vmaps[j]; im=ax[1][j].imshow(np.where(material,vm,np.nan),cmap='cividis',vmax=np.nanpercentile(vm[material],95)); ax[1][j].set_title(f'NBD-FEM var/mean^2 q{a}-{b}',fontsize=9); ax[1][j].axis('off'); plt.colorbar(im,ax=ax[1][j],fraction=0.046)
plt.tight_layout(); save(fig,'02_fluctuation_maps'); plt.show()

## 3) 예상 물질 — (a) 링 패턴에서, (b) 회절 스팟에서

(a) MAX radial + 스팟-반경 히스토그램의 링을 후보 상 d-spacing과 매칭. (b) 밝은 스팟 각각의 |q|=1/d로 후보.
허용오차는 물리 분해능 RING_TOL. **여러 상이 겹치면 모두 표시**(확정 금지).

In [ ]:
from scipy.signal import find_peaks
from scipy.ndimage import white_tophat, maximum_filter
mxc=fds.clean_pattern(np.asarray(cube.max_dp(),float),hot_threshold=HOT_THRESHOLD)
# (a) 링 패턴: MAX radial residual 봉우리 + 스팟-반경 히스토그램
qm,Im=fds.azimuthal_integrate(mxc,(cx,cy),q_per_px=QPP)
_,__,base_m,res_m=fds.find_fsdp(qm,Im,q_lo=0.10,q_hi=float(qm.max()),return_curves=True)
mwin=(qm>=max(q_beam,0.15)); qw,rw=qm[mwin],res_m[mwin]
noise=np.median(np.abs(rw-np.median(rw)))*1.4826+1e-9
pk,pr=find_peaks(rw,prominence=1.2*noise,distance=3)
ring_q=sorted(float(qw[i]) for i in pk)
# 스팟-반경 히스토그램(다결정 powder)
peaks=fds.detect_bragg_peaks(mxc,center=(cx,cy),q_per_px=QPP,min_distance=SPOT_MINDIST,
                             min_prominence_frac=SPOT_PROM,exclude_center_frac=float(min(0.5,q_beam/(QPP*(dp[0]//2)))))
qsp=np.array([p['q'] for p in peaks]); qsp=qsp[(qsp>q_beam)&(qsp<1.15)]
hb=np.arange(float(q_beam),1.15,0.02); hist,edges=np.histogram(qsp,bins=hb); ctr=0.5*(edges[:-1]+edges[1:])
hp,_=find_peaks(hist,height=max(2,0.2*hist.max()),distance=2); ring_q_spot=sorted(float(ctr[i]) for i in hp)
allrings=sorted(set([round(q,3) for q in ring_q]+[round(q,3) for q in ring_q_spot]))
print('링 패턴 q (radial+spot):',[ (round(q,3),round(1/q,2)) for q in allrings])
print('\n(a) 링 패턴에서 예상 물질 (여러 링 맞을수록 신뢰):')
ranked=fds.match_rings(allrings,rings=fds.COMPOUND_RINGS,tol=RING_TOL)
for c,score,matched,missing,mindq in ranked:
    print(f'  {c:7s} score {score:.2f} | matched d={[round(1/q,2) for q in matched]} | missing-strong d={[round(1/q,2) for q in missing]}')

# (b) 밝은 스팟에서 예상 물질
th=white_tophat(mxc,size=11)
yy,xx=np.mgrid[0:dp[0],0:dp[1]]; rr=np.hypot(xx-cx,yy-cy)*QPP
th[(rr<q_beam)|(rr>1.15)]=0
locm=(th==maximum_filter(th,size=2*SPOT_MINDIST+1))&(th>SPOT_PROM*th.max())
ys,xs=np.where(locm)
spots=sorted([dict(x=int(x),y=int(y),q=float(np.hypot(x-cx,y-cy)*QPP),c=float(th[y,x])) for x,y in zip(xs,ys)],key=lambda s:-s['c'])
def which(q):
    d=1.0/q; return sorted({c for c in fds.COMPOUND_RINGS for dd,w in fds.COMPOUND_RINGS[c] if abs(1.0/dd-q)<=RING_TOL})
print(f'\n(b) 밝은 회절 스팟 top10 -> 예상 물질:')
for s in spots[:10]:
    print(f"  (x{s['x']},y{s['y']}) |q|={s['q']:.3f} d={1/s['q']:.2f}A -> {which(s['q'])}")

fig,ax=plt.subplots(1,2,figsize=(14,4.6))
ax[0].plot(qm,res_m,'-',color='0.4',lw=0.9)
for q in allrings: ax[0].axvline(q,color='k',ls=':',lw=1.0)
for c in fds.COMPOUND_RINGS:
    for qc,w in fds.compound_ring_q(c,rings=fds.COMPOUND_RINGS):
        if qc<=qm.max(): ax[0].axvline(qc,color=colors.get(c,'0.5'),ls='--',lw=0.3+0.9*w,alpha=0.6)
    ax[0].plot([],[],color=colors.get(c,'0.5'),lw=2,label=c)
ax[0].set_xlim(0,qm.max()); ax[0].set_xlabel('q (1/A)'); ax[0].set_ylabel('MAX residual'); ax[0].set_title('(a) ring pattern vs candidate rings'); ax[0].legend(fontsize=7,ncol=2)
g=(np.clip(mxc,0,None)/mxc.max())**0.3
ax[1].imshow(g,cmap='gray'); ax[1].plot(cx,cy,'c+',ms=8)
ax[1].scatter([s['x'] for s in spots[:20]],[s['y'] for s in spots[:20]],s=18,facecolors='none',edgecolors='cyan',lw=0.7)
ax[1].set_title('(b) MAX + bright spots (top20)'); ax[1].axis('off')
plt.tight_layout(); save(fig,'03_expected_materials'); plt.show()
save_csv('03_ring_match',['compound','score','matched_d_A','missing_strong_d_A'],
         [[c,f'{score:.3f}','|'.join(str(round(1/q,2)) for q in matched),'|'.join(str(round(1/q,2)) for q in missing)] for c,score,matched,missing,_ in ranked])

## 3b) 비정질 링(halo) — 결정질만이 아니다

§3~§5b는 **MAX**(결정 스팟 모음)라 결정질만 본다. **비정질은 MEAN 패턴의 넓은 FSDP halo**에 있다. 여기서
그 FSDP 반경의 환형 virtual image(두께정규화)로 **비정질이 실공간 어디**에 있는지 + 그 영역 **평균 NBD**를 본다.
MEAN(halo) vs MAX(spots)를 나란히 두면 '결정 vs 비정질'이 분리된다.

In [ ]:
# 비정질 링: MEAN 패턴(결정 MAX 아님)의 FSDP에서 환형 DF -> 비정질 위치 + 그 영역 평균 NBD
q_am = q_ring if (np.isfinite(q_ring) and q_ring>0) else 0.25
DQa=0.05; Tin,Tout=0.15/QPP,1.0/QPP
amorph_img=np.asarray(fds.structural_map(cube,(cx,cy),(q_am-DQa)/QPP,(q_am+DQa)/QPP,Tin,Tout),float)
thrA=np.nanpercentile(amorph_img[material],75); regA=material&(amorph_img>=thrA)
amorph_nbd=np.asarray(fds.average_pattern_aligned(cube,regA),float)
print(f'비정질 FSDP q={q_am:.3f} (d={1/q_am:.2f}A, nn~{1.23/q_am:.2f}A) conf {ring_conf:.1f} | region {int(regA.sum())}pos')
print('  -> 비정질은 넓은 halo라 "링 전체"로만 잡힘(결정 스팟과 대비). 좌1=MEAN halo, 우2=비정질 위치+그 NBD')
cen=(dp[1]/2.0,dp[0]/2.0); Zt=int(min(dp[0]//2,dp[1]//2,1.05/QPP+6))
fig,ax=plt.subplots(1,4,figsize=(18,4.4))
ax[0].imshow(np.log1p(mp),cmap='magma'); ax[0].plot(cx,cy,'c+',ms=8); ax[0].set_title('MEAN pattern (amorphous halo)'); ax[0].axis('off')
ax[1].plot(qd,Id,'k-'); ax[1].axvline(q_am,color='r',ls='--',label=f'FSDP q={q_am:.2f}'); ax[1].axvspan(0,q_beam,color='orange',alpha=0.13)
ax[1].set_xlabel('q (1/A)'); ax[1].set_ylabel('mean I(q)'); ax[1].set_title('mean radial: broad amorphous ring'); ax[1].legend(fontsize=7)
im=ax[2].imshow(np.where(material,amorph_img,np.nan),cmap='inferno'); ax[2].set_title(f'amorphous-ring virtual image\n(FSDP d={1/q_am:.2f}A, thickness-norm)',fontsize=9); ax[2].axis('off'); plt.colorbar(im,ax=ax[2],fraction=0.046)
g=(np.clip(amorph_nbd,0,None)/(np.nanmax(amorph_nbd)+1e-9))**0.3
ax[3].imshow(g,cmap='inferno'); ax[3].plot(*cen,'c+',ms=5); ax[3].add_patch(plt.Circle(cen,q_am/QPP,fill=False,ec='r',ls='--',lw=1))
ax[3].set_xlim(cen[0]-Zt,cen[0]+Zt); ax[3].set_ylim(cen[1]+Zt,cen[1]-Zt); ax[3].axis('off'); ax[3].set_title('amorphous region mean NBD')
plt.tight_layout(); save(fig,'03b_amorphous_ring'); plt.show()
np.save(os.path.join(SAVE_DIR,'03b_amorphous_ring_image.npy'),amorph_img); np.save(os.path.join(SAVE_DIR,'03b_amorphous_nbd.npy'),amorph_nbd)

## 3c) 비정질 halo의 **모든 봉우리**(FSDP + 어깨 + 고q) — 놓친 것 포함 + 후보 검증

FSDP 하나만이 아니라 MEAN 잔차 전 구간에서 **봉우리 + 어깨(shoulder)**를 다 잡는다(0.2 오른쪽 어깨, ~0.8 약한 피크 포함).
**비정질은 결정 Bragg가 아니므로** 위치는 d=1/q 와 nn≈1.23/q 둘 다 표기하고, 각 봉우리를 **후보 상의 강한 링 위치와
느슨히 비교**(비정질 첫 halo ≈ 그 물질의 최강 회절 링 근처)해 '어느 후보가 비정질로 있을 만한가'를 본다.
각 봉우리의 환형 virtual image(두께정규화)로 그 halo 성분이 실공간 어디인지도 본다.

In [ ]:
# MEAN 잔차(§0c의 qd,Id,fsdp_base). 작은 봉우리(0.3/0.4/0.8)가 큰 FSDP 어깨에 묻혀서
# band-pass(넓은 엔벨로프 제거)로 드러냄.
from scipy.signal import find_peaks
q=qd; res_am=Id-fsdp_base
def sm(y,k): k=int(k)|1; return np.convolve(y,np.ones(k)/k,mode='same')
narrow=sm(res_am,5); broad=sm(res_am,35)
fine=narrow-broad                                  # 넓은 FSDP 어깨 제거 -> 미세 봉우리만 남음
mrng=q>q_beam
fwin=(q>q_beam)&(q<0.60); idxa=np.arange(len(q))
fsdp_i=int(idxa[fwin][np.argmax(broad[fwin])])     # 주 FSDP(넓은 봉우리)
nz=np.median(np.abs(fine[mrng]-np.median(fine[mrng])))*1.4826+1e-9
fp,_=find_peaks(fine,prominence=1.0*nz,distance=3)
fp=[int(i) for i in fp if q[i]>q_beam and abs(i-fsdp_i)>4]
allpk=sorted(set([fsdp_i]+fp),key=lambda i:q[i])
def near(qq,tol=0.045):                             # 이 q에 링을 가진 후보(S=강한링, w=약한링)
    return sorted({f"{c}({dd:.2f},{'S' if w>=0.6 else 'w'})" for c,rr in fds.COMPOUND_RINGS.items() for dd,w in rr if abs(1/dd-qq)<=tol})
print('비정질 halo 봉우리 (band-pass) — q | d=1/q | type | 후보 링(S=강,w=약):')
rows=[]
for i in allpk:
    qq=float(q[i]); typ='FSDP' if i==fsdp_i else 'weak'; note='(medium-range order, nn=1.23/q 부적용)' if i==fsdp_i else ''
    cn=near(qq); print(f'  q={qq:.3f}  d={1/qq:.2f}A  [{typ:4s}] {note} -> {cn if cn else "(no ring candidate)"}')
    rows.append([f'{qq:.4f}',f'{1/qq:.3f}',typ,'|'.join(cn)])
save_csv('03c_amorphous_peaks',['q_invA','d_A','type','ring_candidates'],rows)

fig,ax=plt.subplots(1,2,figsize=(15,4.6))
ax[0].plot(q,res_am,'-',color='0.6',lw=0.8,label='residual'); ax[0].plot(q,narrow,'k-',lw=1.0,label='smoothed'); ax[0].plot(q,broad,'g--',lw=1.0,label='broad envelope')
ax[0].axvline(q[fsdp_i],color='r',lw=1.3,label=f'FSDP q={q[fsdp_i]:.3f} (d={1/q[fsdp_i]:.2f}A)'); ax[0].axvspan(0,q_beam,color='orange',alpha=0.12)
ax[0].set_xlabel('q (1/A)'); ax[0].set_ylabel('mean residual'); ax[0].set_title('mean residual: broad FSDP dominates'); ax[0].legend(fontsize=7)
ax[1].plot(q,fine,'k-',lw=1.0); ax[1].axhline(0,color='0.8',lw=0.5)
for i in fp: ax[1].axvline(q[i],color='b',lw=1.1); ax[1].text(q[i],ax[1].get_ylim()[1]*0.9,f'{q[i]:.2f}',color='b',fontsize=6,ha='center')
ax[1].axvspan(0,q_beam,color='orange',alpha=0.12)
for c in fds.COMPOUND_RINGS:
    for dd,w in fds.COMPOUND_RINGS[c]:
        ax[1].axvline(1/dd,color=colors.get(c,'0.5'),ls='--',lw=0.3+0.8*w,alpha=0.45)
    ax[1].plot([],[],color=colors.get(c,'0.5'),lw=2,label=c)
ax[1].set_xlim(q_beam,1.1); ax[1].set_xlabel('q (1/A)'); ax[1].set_ylabel('band-pass (fine bumps)')
ax[1].set_title('faint peaks (blue) revealed by removing FSDP envelope, vs candidate rings'); ax[1].legend(fontsize=6,ncol=3)
plt.tight_layout(); save(fig,'03c_amorphous_peaks'); plt.show()

# 검출 봉우리들의 환형 virtual image(두께정규화; 최대 6개)
sel=sorted(allpk,key=lambda i:q[i])[:6]
Tin,Tout=0.15/QPP,1.0/QPP; DQa=0.030
fig,ax=plt.subplots(1,len(sel),figsize=(3.5*len(sel),3.8),squeeze=False)
for j,i in enumerate(sel):
    qq=float(q[i]); vi=np.asarray(fds.structural_map(cube,(cx,cy),(qq-DQa)/QPP,(qq+DQa)/QPP,Tin,Tout),float)
    im=ax[0][j].imshow(np.where(material,vi,np.nan),cmap='inferno'); ax[0][j].axis('off')
    cn=near(qq); lbl=' / '.join(sorted({x.split('(')[0] for x in cn})) or 'amorphous?'
    ax[0][j].set_title(f'q={qq:.3f} d={1/qq:.2f}A\n[{lbl}]',fontsize=7.5); plt.colorbar(im,ax=ax[0][j],fraction=0.046)
fig.suptitle('amorphous halo-peak virtual images (annular DF / total, thickness-norm)',fontsize=11)
plt.tight_layout(); save(fig,'03c_amorphous_peak_images'); plt.show()

## 4) [링 기반] 각 상: 링 위치(virtual image) + 그 위치 평균 NBD

각 후보 상의 **대표 링**으로 두께정규화 DF를 만들어 **실공간 위치**를 얻고(상위 %를 그 상 영역으로), 그 위치들의
**raw NBD를 빔중심 정렬 평균** → 깨끗한 평균 패턴. 그 평균에서 **선택 링 외 다른 링도 보이면** 그 상이 실제로 존재
한다는 교차증거(§2j ≥2링 논리). 5상 각각.

In [ ]:
DQ=0.03; Tin,Tout=0.2/QPP,1.0/QPP; TOPFRAC=0.25
loc4={}; reg4={}; nbd4={}; qn4={}; res4={}; nmatch4={}
for c in CANDIDATES:
    qc=1.0/STRONG_RING[c]
    m=np.asarray(fds.structural_map(cube,(cx,cy),(qc-DQ)/QPP,(qc+DQ)/QPP,Tin,Tout),float); loc4[c]=m
    thr=np.nanpercentile(m[material],100*(1-TOPFRAC)); reg=material&(m>=thr); reg4[c]=reg
    pat=np.asarray(fds.average_pattern_aligned(cube,reg),float); nbd4[c]=pat
    qn,In=fds.azimuthal_integrate(pat,(cx,cy),q_per_px=QPP)
    _,__,bn,rn=fds.find_fsdp(qn,In,q_lo=0.10,q_hi=float(qn.max()),return_curves=True)
    qn4[c]=qn; res4[c]=rn
    # 평균 NBD에서 이 상의 링이 몇 개나 봉우리로 잡히나(교차확인)
    nm=0
    for dd,w in fds.COMPOUND_RINGS[c]:
        qcc=1.0/dd
        if q_beam<=qcc<=qn.max():
            win=np.abs(qn-qcc)<=RING_TOL
            if win.any() and rn[win].max()>1.5*(np.median(np.abs(rn-np.median(rn)))*1.4826+1e-9): nm+=1
    nmatch4[c]=nm
    print(f'{c:7s} 대표링 d={STRONG_RING[c]:.2f}A -> region {int(reg.sum())}pos | 평균NBD에서 이 상 링 {nm}개 봉우리 {"(>=2 교차확인)" if nm>=2 else ""}')

fig,ax=plt.subplots(2,5,figsize=(22,8))
for j,c in enumerate(CANDIDATES):
    im=ax[0][j].imshow(np.where(material,loc4[c],np.nan),cmap='inferno'); ax[0][j].set_title(f'{c} location\n(ring d={STRONG_RING[c]:.2f}A)',fontsize=9); ax[0][j].axis('off'); plt.colorbar(im,ax=ax[0][j],fraction=0.046)
    cen=(dp[1]/2.0,dp[0]/2.0); Zt=int(min(dp[0]//2,dp[1]//2,1.05/QPP+6))
    pat=nbd4[c]; g=(np.clip(pat,0,None)/(np.nanmax(pat)+1e-9))**0.3
    a=ax[1][j]; a.imshow(g,cmap='inferno'); a.plot(*cen,'c+',ms=6)
    for dd,w in fds.COMPOUND_RINGS[c]:
        a.add_patch(plt.Circle(cen,(1.0/dd)/QPP,fill=False,ec='cyan',ls='--',lw=0.4+0.9*w,alpha=0.6))
    a.set_xlim(cen[0]-Zt,cen[0]+Zt); a.set_ylim(cen[1]+Zt,cen[1]-Zt); a.axis('off')
    a.set_title(f'{c} mean NBD (2D)\n{nmatch4[c]} of its rings as peaks',fontsize=8)
fig.suptitle('TASK4 [ring-based]: per-phase location (top) + mean NBD from those positions (bottom)',fontsize=12)
plt.tight_layout(); save(fig,'04_ring_location_meanNBD'); plt.show()
for c in CANDIDATES: np.save(os.path.join(SAVE_DIR,f'04_meanNBD_{c}.npy'), nbd4[c])
save_csv('04_ring_crosscheck',['compound','ring_d_A','region_positions','rings_as_peaks','cross_confirmed_ge2'],
         [[c,f'{STRONG_RING[c]:.2f}',int(reg4[c].sum()),nmatch4[c],'yes' if nmatch4[c]>=2 else 'no'] for c in CANDIDATES])

## 5) [피크 기반] 각 상: 대표 스팟 위치 + 그 위치 평균 NBD

각 상에 대해 **그 상의 링과 |q|가 맞는 가장 밝은 회절 스팟**을 골라 2D 가우시안으로 정밀 피팅 → 조리개 DF로 위치 →
상위 %를 영역으로 → 정렬 평균 NBD. 맞는 스팟이 없으면 '해당 상 밝은 스팟 없음'으로 표시(결정질 근거 약함).

In [ ]:
from scipy.optimize import curve_fit
def g2d(P,A,mx,my,s,cc): X,Y=P; return (A*np.exp(-((X-mx)**2+(Y-my)**2)/(2*s**2))+cc).ravel()
def fit_spot(x0,y0,W=6):
    xa,xb=max(0,x0-W),min(dp[1],x0+W+1); ya,yb=max(0,y0-W),min(dp[0],y0+W+1)
    sub=mxc[ya:yb,xa:xb].astype(float); gy,gx=np.mgrid[ya:yb,xa:xb]
    try:
        p,_=curve_fit(g2d,(gx,gy),sub.ravel(),p0=[sub.max()-sub.min(),x0,y0,2.0,float(np.median(sub))],maxfev=20000)
        return float(p[1]),float(p[2]),abs(float(p[3])),True
    except Exception: return float(x0),float(y0),2.0,False
yy,xx=np.mgrid[0:dp[0],0:dp[1]]
loc5={}; nbd5={}; qn5={}; res5={}; pick5={}
for c in CANDIDATES:
    ringqs=[1.0/d for d,_ in fds.COMPOUND_RINGS[c]]
    cand=[s for s in spots if min(abs(s['q']-rq) for rq in ringqs)<=RING_TOL]
    if not cand: pick5[c]=None; print(f'{c:7s}: 맞는 밝은 스팟 없음 (결정질 근거 약함)'); continue
    s=max(cand,key=lambda s:s['c'])                       # 그 중 가장 밝은
    mxf,myf,sf,ok=fit_spot(s['x'],s['y']); qf=float(np.hypot(mxf-cx,myf-cy)*QPP)
    ap=max(SPOT_AP,2*sf); mask=((xx-mxf)**2+(yy-myf)**2)<=ap**2
    vi=np.asarray(fds.virtual_image(cube,mask),float); loc5[c]=vi
    thr=np.nanpercentile(vi[material],75); reg=material&(vi>=thr)
    pat=np.asarray(fds.average_pattern_aligned(cube,reg),float); nbd5[c]=pat
    qn,In=fds.azimuthal_integrate(pat,(cx,cy),q_per_px=QPP)
    _,__,bn,rn=fds.find_fsdp(qn,In,q_lo=0.10,q_hi=float(qn.max()),return_curves=True); qn5[c]=qn; res5[c]=rn
    pick5[c]=dict(x=mxf,y=myf,sigma=sf,q=qf,d=1.0/qf,region=int(reg.sum()),ok=ok)
    print(f"{c:7s}: spot (x{mxf:.1f},y{myf:.1f}) sigma={sf:.2f} d={1/qf:.2f}A region {int(reg.sum())}pos")

have=[c for c in CANDIDATES if pick5.get(c)]
n=len(have)
if n:
    fig,ax=plt.subplots(2,n,figsize=(4.3*n,8),squeeze=False)
    for j,c in enumerate(have):
        im=ax[0][j].imshow(np.where(material,loc5[c],np.nan),cmap='inferno'); ax[0][j].set_title(f"{c} location\n(spot d={pick5[c]['d']:.2f}A)",fontsize=9); ax[0][j].axis('off'); plt.colorbar(im,ax=ax[0][j],fraction=0.046)
        cen=(dp[1]/2.0,dp[0]/2.0); Zt=int(min(dp[0]//2,dp[1]//2,1.05/QPP+6))
        pat=nbd5[c]; g=(np.clip(pat,0,None)/(np.nanmax(pat)+1e-9))**0.3
        a=ax[1][j]; a.imshow(g,cmap='inferno'); a.plot(*cen,'c+',ms=6)
        for dd,w in fds.COMPOUND_RINGS[c]:
            a.add_patch(plt.Circle(cen,(1.0/dd)/QPP,fill=False,ec='0.6',ls='--',lw=0.4+0.9*w,alpha=0.6))
        a.add_patch(plt.Circle(cen,pick5[c]['q']/QPP,fill=False,ec='r',lw=1.3))
        a.set_xlim(cen[0]-Zt,cen[0]+Zt); a.set_ylim(cen[1]+Zt,cen[1]-Zt); a.axis('off')
        a.set_title(f'{c} mean NBD (2D)\nred=selected ring',fontsize=8)
    fig.suptitle('TASK5 [peak-based]: per-phase location (top) + mean NBD (bottom); red=fitted spot q',fontsize=12)
    plt.tight_layout(); save(fig,'05_peak_location_meanNBD'); plt.show()
    for c in have: np.save(os.path.join(SAVE_DIR,f'05_meanNBD_{c}.npy'), nbd5[c])
else:
    print('맞는 밝은 스팟이 있는 상이 없음 — 결정질 스팟 근거 약함(비정질 우세).')
save_csv('05_peak_pick',['compound','has_spot','x','y','sigma_px','d_A','region_positions'],
         [[c,'yes' if pick5.get(c) else 'no',
           f"{pick5[c]['x']:.1f}" if pick5.get(c) else '', f"{pick5[c]['y']:.1f}" if pick5.get(c) else '',
           f"{pick5[c]['sigma']:.2f}" if pick5.get(c) else '', f"{pick5[c]['d']:.2f}" if pick5.get(c) else '',
           pick5[c]['region'] if pick5.get(c) else 0] for c in CANDIDATES])

## 5b) MAX radial 5개 봉우리 → 5개 링 + 각 링 밝은 스팟 4곳 (각각 virtual image + 평균 NBD)

MAX radial 적분에서 중심빔을 뺀 **상위 5개 봉우리**(=5개 다결정 링). 각 링마다 **(1) 링 전체 virtual image + 그
영역 평균 NBD**, 그리고 **(2) 그 링 위에서 방위각으로 멀리 떨어진 스팟 4곳 각각의 virtual image + 각 스팟 영역 평균 NBD**. 즉 5링 + 20스팟
모두 **하나의 합친 NBD가 아니라 각자의 평균 NBD**를 보여준다(링/스팟 모두 총산란 나눠 두께정규화).
평균 NBD에서 다른 링도 같이 보이면 그 상이 실제로 그 위치에 있다는 교차증거. **주의**: 실데이터(512x512)에서 정렬평균
25회라 수 분 걸릴 수 있음 — 느리면 `TOPFRAC_SPOT`/스팟 수를 줄이세요.

In [ ]:
from scipy.signal import find_peaks
from scipy.ndimage import white_tophat, maximum_filter
mxc=fds.clean_pattern(np.asarray(cube.max_dp(),float),hot_threshold=HOT_THRESHOLD)
qm,Im=fds.azimuthal_integrate(mxc,(cx,cy),q_per_px=QPP)
_,__,base_m,res_m=fds.find_fsdp(qm,Im,q_lo=0.10,q_hi=float(qm.max()),return_curves=True)
mwin=(qm>=max(q_beam,0.15)); qw,rw=qm[mwin],res_m[mwin]
noise=np.median(np.abs(rw-np.median(rw)))*1.4826+1e-9
pk,props=find_peaks(rw,prominence=1.0*noise,distance=3)
order=np.argsort(props['prominences'])[::-1]
ring5=sorted(float(qw[pk[i]]) for i in order[:5])                 # MAX radial 상위 5봉우리(=링)
def which(q): return sorted({c for c in fds.COMPOUND_RINGS for dd,w in fds.COMPOUND_RINGS[c] if abs(1.0/dd-q)<=RING_TOL})
print(f'MAX radial 상위 {len(ring5)}개 봉우리:')
for q in ring5: print(f'  q={q:.3f} d={1/q:.2f}A -> 예상 {which(q)}')

# --- 강건 스팟 검출: top-hat(halo 제거) 후 median+SPOT_NMAD*MAD 이상만 = 진짜 Bragg 스팟(노이즈 배제) ---
yy,xx=np.mgrid[0:dp[0],0:dp[1]]; rr=np.hypot(xx-cx,yy-cy)*QPP
th=white_tophat(mxc,size=11); th[(rr<q_beam)|(rr>1.15)]=0
vv=th[(rr>=q_beam)&(rr<=1.15)]; med=np.median(vv); mad=1.4826*np.median(np.abs(vv-med))+1e-9
thresh=med+SPOT_NMAD*mad
locm=(th==maximum_filter(th,size=2*SPOT_MINDIST+1))&(th>thresh)
sy,sx=np.where(locm)
spots_all=sorted([dict(x=int(x),y=int(y),q=float(np.hypot(x-cx,y-cy)*QPP),c=float(th[y,x])) for x,y in zip(sx,sy)],key=lambda s:-s['c'])
print(f'강건 임계(median+{SPOT_NMAD}*MAD) 통과 스팟: {len(spots_all)}개')

T=np.asarray(fds.annular_dark_field(cube,center=(cx,cy),r_inner=0.15/QPP,r_outer=1.0/QPP),float); T=np.where(T>0,T,np.nan)
DQ=0.03; TOPFRAC_RING=0.25; TOPFRAC_SPOT=0.10
cen=(dp[1]/2.0,dp[0]/2.0); Zt=int(min(dp[0]//2,dp[1]//2,1.05/QPP+6))
def region_nbd(img,frac): thr=np.nanpercentile(img[material],100*(1-frac)); reg=material&(img>=thr); return np.asarray(fds.average_pattern_aligned(cube,reg),float)
def draw_nbd(a,pat,refc=None,sel=None):
    g=(np.clip(pat,0,None)/(np.nanmax(pat)+1e-9))**0.3; a.imshow(g,cmap='inferno'); a.plot(*cen,'c+',ms=4)
    if refc:
        for dd,w in fds.COMPOUND_RINGS[refc]: a.add_patch(plt.Circle(cen,(1.0/dd)/QPP,fill=False,ec='0.6',ls='--',lw=0.4+0.8*w,alpha=0.6))
    if sel: a.add_patch(plt.Circle(cen,sel/QPP,fill=False,ec='r',lw=1.1))
    a.set_xlim(cen[0]-Zt,cen[0]+Zt); a.set_ylim(cen[1]+Zt,cen[1]-Zt); a.axis('off')

ring_imgs=[];ring_nbd=[];spot_info=[];spot_imgs=[];spot_nbd=[]
for q in ring5:
    ri=np.asarray(fds.annular_dark_field(cube,center=(cx,cy),r_inner=(q-DQ)/QPP,r_outer=(q+DQ)/QPP),float)/T
    ring_imgs.append(ri); ring_nbd.append(region_nbd(ri,TOPFRAC_RING))
    # 이 링 위에서 '가장 밝은 4개' — 대칭 강제 X, 밝기 우선 + 중복(같은 스팟) 제거
    on=[s for s in spots_all if abs(s['q']-q)<=DQ]      # spots_all은 이미 밝기순
    picked=[]
    for s in on:
        if all((s['x']-p['x'])**2+(s['y']-p['y'])**2>=(3*SPOT_MINDIST)**2 for p in picked): picked.append(s)
        if len(picked)>=4: break
    spot_info.append(picked); imgs=[];nbds=[]
    for s in picked:
        mask=((xx-s['x'])**2+(yy-s['y'])**2)<=SPOT_AP**2
        vi=np.asarray(fds.virtual_image(cube,mask),float)/T; imgs.append(vi); nbds.append(region_nbd(vi,TOPFRAC_SPOT))
    spot_imgs.append(imgs); spot_nbd.append(nbds)
    print(f'  ring d={1/q:.2f}A: 밝은 스팟 {len(picked)}개 (밝기순, 중복제거)')

# A) overview
fig,axo=plt.subplots(1,1,figsize=(7.5,7.5))
axo.imshow((np.clip(mxc,0,None)/mxc.max())**0.3,cmap='gray'); axo.plot(cx,cy,'c+',ms=10)
for k,q in enumerate(ring5):
    axo.add_patch(plt.Circle((cx,cy),q/QPP,fill=False,ec='yellow',ls='--',lw=0.8))
    for j,s in enumerate(spot_info[k]):
        axo.add_patch(plt.Circle((s['x'],s['y']),SPOT_AP,fill=False,ec='cyan',lw=1.3)); axo.text(s['x']+SPOT_AP+1,s['y'],f'{k+1}.{j+1}',color='cyan',fontsize=6,va='center')
axo.set_title('MAX: 5 rings (yellow) + 4 brightest real spots each (cyan)'); axo.axis('off'); plt.tight_layout(); save(fig,'07_max_rings_spots_overview'); plt.show()

# B) 5 rings: virtual image + mean NBD
nR=len(ring5)
fig,ax=plt.subplots(2,nR,figsize=(4.0*nR,8),squeeze=False)
for k,q in enumerate(ring5):
    lbl=' / '.join(which(q)) or 'unindexed'
    im=ax[0][k].imshow(np.where(material,ring_imgs[k],np.nan),cmap='inferno'); ax[0][k].axis('off'); ax[0][k].set_title(f'ring{k+1} q={q:.3f}\nd={1/q:.2f}A [{lbl}]',fontsize=8); plt.colorbar(im,ax=ax[0][k],fraction=0.046)
    draw_nbd(ax[1][k],ring_nbd[k],refc=(which(q)[0] if which(q) else None),sel=q); ax[1][k].set_title(f'ring{k+1} mean NBD',fontsize=8)
fig.suptitle('5 rings: virtual image (top) + mean NBD at that region (bottom)',fontsize=12); plt.tight_layout(); save(fig,'07_ring_image_and_NBD'); plt.show()

# C) spot virtual images
fig,ax=plt.subplots(nR,4,figsize=(16,3.8*nR),squeeze=False)
for k,q in enumerate(ring5):
    lbl=' / '.join(which(q)) or 'unindexed'
    for j in range(4):
        a=ax[k][j]
        if j<len(spot_imgs[k]):
            s=spot_info[k][j]; im=a.imshow(np.where(material,spot_imgs[k][j],np.nan),cmap='inferno'); a.set_title(f"r{k+1} d={1/q:.2f} spot{j+1}\n(x{s['x']},y{s['y']}) [{lbl}]",fontsize=7)
        else: a.set_title(f'r{k+1} spot{j+1}: none',fontsize=7)
        a.axis('off')
fig.suptitle('single-spot virtual images (5 rings x up to 4 brightest spots, thickness-norm)',fontsize=13); plt.tight_layout(); save(fig,'07_spot_virtual_images_20'); plt.show()

# D) spot mean NBDs
fig,ax=plt.subplots(nR,4,figsize=(16,3.8*nR),squeeze=False)
for k,q in enumerate(ring5):
    for j in range(4):
        a=ax[k][j]
        if j<len(spot_nbd[k]): draw_nbd(a,spot_nbd[k][j],refc=(which(q)[0] if which(q) else None),sel=q); a.set_title(f'r{k+1} d={1/q:.2f} spot{j+1} NBD',fontsize=7)
        else: a.set_title(f'r{k+1} spot{j+1}: none',fontsize=7); a.axis('off')
fig.suptitle('single-spot mean NBD patterns (each spot region averaged)',fontsize=13); plt.tight_layout(); save(fig,'07_spot_NBD_20'); plt.show()

rows=[]
for k,q in enumerate(ring5):
    for j,s in enumerate(spot_info[k]): rows.append([k+1,f'{q:.4f}',f'{1/q:.3f}',j+1,s['x'],s['y'],f"{s['q']:.4f}",'|'.join(which(q)) or 'unindexed'])
save_csv('07_rings_spots',['ring','ring_q_invA','ring_d_A','spot','x','y','spot_q_invA','expected'],rows)
for k,q in enumerate(ring5):
    np.save(os.path.join(SAVE_DIR,f'07_ring{k+1}_d{1/q:.2f}_image.npy'),ring_imgs[k]); np.save(os.path.join(SAVE_DIR,f'07_ring{k+1}_d{1/q:.2f}_NBD.npy'),ring_nbd[k])

## 5c) 전체 Bragg 스팟 census — 많은 스팟이 다 뭔가? (LiF / 다른상 / 공유 / 미설명)

회절 패턴엔 스팟이 수백 개(=grain 수백 개). 각 스팟의 d=1/q로 **전부 분류**한다: 그 d가 **한 상에만 고유**하면
그 상, 여러 상이면 **shared**, 어느 후보도 아니면 **unexplained**. 몇 %가 LiF인지/다른 결정인지/미설명인지
정량화하고, **비-LiF·미설명 스팟은 DF로 이미지**해 진짜 다른 grain인지 본다(다른 상 사냥).

In [ ]:
from scipy.ndimage import white_tophat, maximum_filter
from scipy.signal import find_peaks
from collections import Counter
mxc=fds.clean_pattern(np.asarray(cube.max_dp(),float),hot_threshold=HOT_THRESHOLD)
yy,xx=np.mgrid[0:dp[0],0:dp[1]]; rr=np.hypot(xx-cx,yy-cy)*QPP
# 링(ring5) 오버레이용 재계산
qm,Im=fds.azimuthal_integrate(mxc,(cx,cy),q_per_px=QPP)
_,__,bm,rm=fds.find_fsdp(qm,Im,q_lo=0.10,q_hi=float(qm.max()),return_curves=True)
mw=(qm>=max(q_beam,0.15)); qw2,rw2=qm[mw],rm[mw]
nz=np.median(np.abs(rw2-np.median(rw2)))*1.4826+1e-9
pkr,pr=find_peaks(rw2,prominence=1.0*nz,distance=3); ring5=sorted(float(qw2[i]) for i in np.argsort(pr['prominences'])[::-1][:5])
# 강건 스팟 검출(median+SPOT_NMAD*MAD)
th=white_tophat(mxc,size=11); th[(rr<q_beam)|(rr>1.15)]=0
vv=th[(rr>=q_beam)&(rr<=1.15)]; med=np.median(vv); mad=1.4826*np.median(np.abs(vv-med))+1e-9
loc=(th==maximum_filter(th,size=2*SPOT_MINDIST+1))&(th>med+SPOT_NMAD*mad)
sy,sx=np.where(loc)
spots=[dict(x=int(x),y=int(y),q=float(np.hypot(x-cx,y-cy)*QPP),c=float(th[y,x])) for x,y in zip(sx,sy)]
def classify(q,tol=0.045):
    hits=sorted({c for c,rr2 in fds.COMPOUND_RINGS.items() for dd,w in rr2 if abs(1/dd-q)<=tol})
    if not hits: return 'unexplained',hits
    if len(hits)==1: return f'{hits[0]}-only',hits
    return 'shared',hits
for s in spots: s['cat'],s['hits']=classify(s['q'])
cats=Counter(s['cat'] for s in spots); ntot=max(len(spots),1)
print(f'전체 Bragg 스팟(grain): {len(spots)}개')
for k,v in cats.most_common(): print(f'  {k:14s}: {v:4d}개 ({100*v/ntot:.0f}%)')
lif_only=[s for s in spots if s['cat']=='LiF-only']
other_only=[s for s in spots if s['cat'].endswith('-only') and s['cat']!='LiF-only']
unexpl=[s for s in spots if s['cat']=='unexplained']
print(f"\n-> LiF 단독 {len(lif_only)} | 다른상 단독 {len(other_only)} | 공유 {cats.get('shared',0)} | 미설명 {len(unexpl)}")
save_csv('05c_spot_census',['x','y','q_invA','d_A','category','candidates'],
         [[s['x'],s['y'],f"{s['q']:.4f}",f"{1/s['q']:.3f}",s['cat'],'|'.join(s['hits'])] for s in spots])

# 잘 보이는 색 + ring5 오버레이 (공유=cyan, 미설명=magenta 로 배경과 대비)
COL={'LiF-only':'red','shared':'cyan','unexplained':'magenta'}
def col(s): return COL.get(s['cat'],'orange')   # 다른상-only=orange
fig,ax=plt.subplots(1,2,figsize=(15,5.4))
ax[0].imshow((np.clip(mxc,0,None)/mxc.max())**0.3,cmap='gray')
for q in ring5: ax[0].add_patch(plt.Circle((cx,cy),q/QPP,fill=False,ec='yellow',ls='--',lw=0.6,alpha=0.7))
for s in spots: ax[0].scatter(s['x'],s['y'],s=26,facecolors='none',edgecolors=col(s),lw=1.1)
ax[0].plot(cx,cy,'w+',ms=9)
for lab,co in [('LiF-only','red'),('other-only','orange'),('shared','cyan'),('unexplained','magenta')]:
    ax[0].scatter([],[],facecolors='none',edgecolors=co,label=lab)
ax[0].legend(fontsize=7,loc='upper right'); ax[0].set_title(f'all {len(spots)} spots on rings, colored by identity'); ax[0].axis('off')
bins=np.arange(1.0,5.0,0.06)
for lab,co in [('LiF-only','red'),('shared','cyan'),('unexplained','magenta')]:
    dd=[1/s['q'] for s in spots if s['cat']==lab]
    if dd: ax[1].hist(dd,bins=bins,color=co,alpha=0.75,label=lab)
oo=[1/s['q'] for s in spots if s['cat'].endswith('-only') and s['cat']!='LiF-only']
if oo: ax[1].hist(oo,bins=bins,color='orange',alpha=0.75,label='other-only')
ax[1].set_xlabel('d = 1/q (A)'); ax[1].set_ylabel('# spots'); ax[1].set_title('spot d-spacing census (colored by identity)'); ax[1].legend(fontsize=7)
plt.tight_layout(); save(fig,'05c_spot_census'); plt.show()

# 비-LiF/미설명 대표 스팟 DF(진짜 다른 grain?) — 최대 4개
T=np.asarray(fds.annular_dark_field(cube,center=(cx,cy),r_inner=0.15/QPP,r_outer=1.0/QPP),float); T=np.where(T>0,T,np.nan)
hunt=sorted(other_only+unexpl,key=lambda s:-s['c'])[:4]
if hunt:
    fig,ax=plt.subplots(1,len(hunt),figsize=(4.0*len(hunt),3.8),squeeze=False)
    for j,s in enumerate(hunt):
        mask=((xx-s['x'])**2+(yy-s['y'])**2)<=SPOT_AP**2
        vi=np.asarray(fds.virtual_image(cube,mask),float)/T
        im=ax[0][j].imshow(np.where(material,vi,np.nan),cmap='inferno'); ax[0][j].axis('off')
        ax[0][j].set_title(f"d={1/s['q']:.2f}A {s['cat']}\n{s['hits'] or 'none'}",fontsize=7.5); plt.colorbar(im,ax=ax[0][j],fraction=0.046)
    fig.suptitle('non-LiF / unexplained spots -> DF (localized grain = real other phase?)',fontsize=11)
    plt.tight_layout(); save(fig,'05c_nonLiF_spot_DF'); plt.show()
else:
    print('비-LiF/미설명 단독 스팟 없음 (거의 다 LiF 또는 공유).')

## 5d) 다이렉트 빔 **근처 저-q 링** 조사 — 빔 컷이 가린 큰-d 링?

지금까지 검출은 `q_beam`(≈beam 가장자리) 안쪽을 다 제외했다. 하지만 **빔 디스크 바로 바깥(작은 q = 큰 d)**
에 다결정 링이 있으면 그게 잘렸을 수 있다. 여기서는 **빔 디스크(α/λ) 바로 바깥부터** 다시 스팟/링을 찾고, 있으면
d-spacing과 virtual image를 본다. **주의**: 큰-d(>4Å) 링은 후보 5상 밖 → 큰 단위셀 상/유기 SEI/초격자 가능,
또는 빔 디스크 가장자리 아티팩트일 수도 있으니 국소 grain으로 뜨는지(진짜 결정)까지 확인한다.

In [ ]:
from scipy.ndimage import white_tophat, maximum_filter
from scipy.signal import find_peaks
mxc=fds.clean_pattern(np.asarray(cube.max_dp(),float),hot_threshold=HOT_THRESHOLD)
yy,xx=np.mgrid[0:dp[0],0:dp[1]]; rr=np.hypot(xx-cx,yy-cy)*QPP
LAMBDA_A=0.0251                                  # 200kV 전자파장(Å)
q_disk=1.1e-3/LAMBDA_A                            # 반수렴각 1.1mrad -> 빔 디스크 q반경(1/A)
q_in=max(1.5*q_disk,0.06); q_hi=0.30             # 빔 디스크 바로 바깥 ~ 0.30
print(f'빔 디스크 q반경~{q_disk:.3f} | near-beam 조사 구간 q={q_in:.3f}~{q_hi} (기존 q_beam={q_beam:.3f})')
th=white_tophat(mxc,size=11); th[(rr<q_in)|(rr>q_hi)]=0
vv=th[(rr>=q_in)&(rr<=q_hi)]; med=np.median(vv); mad=1.4826*np.median(np.abs(vv-med))+1e-9
loc=(th==maximum_filter(th,size=2*SPOT_MINDIST+1))&(th>med+SPOT_NMAD*mad)
sy,sx=np.where(loc)
low=[dict(x=int(x),y=int(y),q=float(np.hypot(x-cx,y-cy)*QPP)) for x,y in zip(sx,sy)]
print(f'near-beam 스팟: {len(low)}개')
lowrings=[]
if low:
    qs=np.array([s['q'] for s in low]); hb=np.arange(q_in,q_hi,0.015); h,e=np.histogram(qs,bins=hb); ct=0.5*(e[:-1]+e[1:])
    pk,_=find_peaks(h,height=max(2,0.3*h.max())); lowrings=sorted(float(ct[i]) for i in pk)
    print('near-beam 링 (q | d=1/q | 후보):')
    for q in lowrings:
        cand=sorted({c for c in fds.COMPOUND_RINGS for dd,w in fds.COMPOUND_RINGS[c] if abs(1/dd-q)<=RING_TOL})
        note=cand if cand else '(후보 밖: 큰 d -> 큰 단위셀/유기/초격자 or 빔 아티팩트)'
        print(f'  q={q:.3f}  d={1/q:.2f}A  -> {note}')
    save_csv('05d_near_beam',['q_invA','d_A'],[[f'{q:.4f}',f'{1/q:.3f}'] for q in lowrings])

qm,Im=fds.azimuthal_integrate(mxc,(cx,cy),q_per_px=QPP)
fig,ax=plt.subplots(1,3,figsize=(16,4.6))
zt=int(q_hi/QPP*1.15)
ax[0].imshow((np.clip(mxc,0,None)/mxc.max())**0.3,cmap='gray')
ax[0].add_patch(plt.Circle((cx,cy),q_in/QPP,fill=False,ec='orange',ls='--',lw=1.2))
ax[0].add_patch(plt.Circle((cx,cy),q_beam/QPP,fill=False,ec='red',ls=':',lw=1.2))
for s in low: ax[0].scatter(s['x'],s['y'],s=22,facecolors='none',edgecolors='lime',lw=1.1)
ax[0].plot(cx,cy,'c+',ms=9); ax[0].set_xlim(cx-zt,cx+zt); ax[0].set_ylim(cy+zt,cy-zt)
ax[0].set_title('near-beam zoom: green=low-q spots\norange--=q_in, red:=old q_beam'); ax[0].axis('off')
ax[1].semilogy(qm,np.clip(Im,1e-2,None),'k-'); ax[1].axvspan(0,q_in,color='orange',alpha=0.15,label=f'excluded q<{q_in:.2f}')
ax[1].axvline(q_beam,color='r',ls=':',label=f'old q_beam={q_beam:.2f}')
for q in lowrings: ax[1].axvline(q,color='lime',lw=1.0)
ax[1].set_xlim(0,0.40); ax[1].set_xlabel('q (1/A)'); ax[1].set_ylabel('MAX I(q)'); ax[1].set_title('low-q radial: ring below old q_beam?'); ax[1].legend(fontsize=7)
if lowrings:
    q0=lowrings[0]; Tin,Tout=0.15/QPP,1.0/QPP
    vi=np.asarray(fds.structural_map(cube,(cx,cy),max(1,(q0-0.02)/QPP),(q0+0.02)/QPP,Tin,Tout),float)
    im=ax[2].imshow(np.where(material,vi,np.nan),cmap='inferno'); plt.colorbar(im,ax=ax[2],fraction=0.046)
    ax[2].set_title(f'near-beam ring virtual image\nq={q0:.3f} d={1/q0:.2f}A (localized grain=real?)'); ax[2].axis('off')
else:
    ax[2].axis('off'); ax[2].set_title('no distinct near-beam ring found')
plt.tight_layout(); save(fig,'05d_near_beam_ring'); plt.show()

## 6) 요약 — 후보표 + 정직한 한계

In [ ]:
print('=== nb6-1 요약 ===')
print('후보 상별 증거 (링 교차확인 / 밝은 스팟 유무):')
for c in CANDIDATES:
    nm=nmatch4.get(c,0); hasp=bool(pick5.get(c))
    verdict='결정질 근거 있음' if (nm>=2 or hasp) else '비정질/약함'
    print(f'  {c:7s}: 평균NBD 링 {nm}개, 밝은스팟 {"O" if hasp else "X"} -> {verdict}')
print('\n[정직한 한계]')
print(f'  - 수렴각 ~1.1 mrad -> q분해능 ~0.09 1/A. d~2A 상(LiF/Li2O/Li3N)은 물리적으로 분리 불가.')
print('  - 라벨의 "예상 물질"은 후보. 회절만으로 확정 가능한 것은 없음(EDS의 F 덩어리로 LiF만 교차확인).')
print('  - EDS(um 시야) vs 4D-STEM(200nm) 스케일 달라 픽셀 정합 불가 — 집단 수준 연결만.')
print('  - 상세: docs/SEI_phase_analysis_report.md 참조.')